# Classical Speed-Density Models: Validation Benchmark for SR-210 Mesa ABM

This notebook encodes the three canonical macroscopic traffic flow models — **Greenshields (1935)**, **Greenberg (1959)**, and **Underwood (1961)** — using parameter values drawn from published empirical studies. It serves three purposes:

1. **Encode and visualize** the three models with literature-derived parameter ranges, including sensitivity analysis across physically plausible parameter values.
2. **Characterize parameter uncertainty** via Monte Carlo sampling across cross-study estimates, producing publication-quality figures with 95% CI bands.
3. **Provide the ABM comparison harness**: once Tier 2 trajectory data from the SR-210 Mesa simulation is available, this notebook extracts the macroscopic fundamental diagram via Edie's generalized definitions and fits the three models via nonlinear least squares.

**No published study has calibrated these models on a steep two-lane mountain road.** The literature spans freeways (Drake 1967), urban arterials (Lu & Meng 2013), expressways (Romanowska 2021), and one two-lane road (Tiwari 2014 Nepal). SR-210 is distinct: 12.4 miles, grades up to 11%, 25–50 mph speed limits, seasonal closure patterns. This notebook produces what may be the first fundamental diagram characterization of a facility like SR-210.

In [ ]:
import os
import subprocess
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import norm
import numpy as np


warnings.filterwarnings('ignore', category=RuntimeWarning)
np.random.seed(42)

# --- project root ---
PROJECT_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

%load_ext autoreload
%autoreload 2

import empirical_benchmark.empirical_benchmark_helpers as ebh 
# --- output directory ---
Path("notebooks/figures").mkdir(exist_ok=True)

# --- style constants ---
FIG_W_SINGLE = (10, 6)
FIG_W_WIDE   = (14, 6)
FIG_W_TRIPLE = (18, 5)
GRID_ALPHA   = 0.3
CI_ALPHA     = 0.20
N_MC         = 2000
CI_LO, CI_HI = 2.5, 97.5

# --- density axis for all plots ---
K_PLOT  = np.linspace(0.5, 160, 600)   # veh/mi (start at 0.5 to avoid Greenberg log singularity)
K_RANGE = np.linspace(0.5, 160, 500)   # coarser version for sensitivity sweeps

sns.set_style("whitegrid")
print("Setup complete.")



---
## 1. Unit Conversions

The five literature sources span two unit systems: **imperial** (Drake 1967 uses mph and veh/mi) and **metric** (Lu & Meng 2013, Romanowska 2021, and Tiwari 2014 use km/h and veh/km). All parameter values are normalized to **mph** and **veh/mi** for consistency with the Mesa ABM output, which reports speed in mph and can produce density in veh/mi from trajectory data.

Conversions are defined inline so this notebook is fully self-contained — no project imports required for this section.

---
## 2. Model Equations

The three models form a natural historical trilogy (1935–1959–1961) that together span the full density range through **complementary failures**:

| Model | Equation | Parameters | v → v_f at k=0? | v → 0 at finite k_j? |
|---|---|---|---|---|
| **Greenshields (1935)** | $v = v_f\left(1 - \frac{k}{k_j}\right)$ | $v_f$, $k_j$ | ✓ Yes | ✓ Yes |
| **Greenberg (1959)** | $v = v_0 \ln\left(\frac{k_j}{k}\right)$ | $v_0$, $k_j$ | ✗ No ($v \to \infty$) | ✓ Yes |
| **Underwood (1961)** | $v = v_f \exp\left(-\frac{k}{k_0}\right)$ | $v_f$, $k_0$ | ✓ Yes | ✗ No ($k_j \to \infty$) |

**Origins:**
- Greenshields derived the linear model from ~7 data points collected with 16mm film cameras (1935, Highway Research Board Proc. Vol. 14). Simple OLS fitting applies directly.
- Greenberg derived the logarithmic model from a hydrodynamic analogy treating traffic as compressible fluid, calibrated on Lincoln Tunnel data (*Operations Research*, 1959, Vol. 7). It diverges as $k \to 0$ — a physical singularity requiring `np.clip(k, 1e-6, None)` in code.
- Underwood derived the exponential from Merritt Parkway data (Yale Bureau of Highway Traffic, 1961) — a limited-access two-lane facility, the closest road-type analog to SR-210 among the three origins.

Each model implies a parabolic flow-density curve $q = kv$ with a single capacity peak. Critical-point derivations:
- **Greenshields:** $k_c = k_j/2$, $q_{\max} = v_f k_j / 4$
- **Greenberg:** $k_c = k_j/e$, $q_{\max} = v_0 k_j / e$
- **Underwood:** $k_c = k_0$, $q_{\max} = v_f k_0 / e$

In [ ]:
# --- parameter glossary ---
print("v_f  : Free-Flow Speed (mph)    — vehicle speed as density approaches zero")
print("k_j  : Jam Density (veh/mi)     — maximum density at which traffic is fully stopped")
print("k_0  : Optimal Density (veh/mi) — density at which flow is maximized (Underwood only)")
print("v_0  : Speed Constant (mph)     — speed scale in the Greenberg logarithmic model")

---
## 3. Literature Parameter Registry

Five published studies provide calibrated parameter values for the three models. Each is described below:

- **Drake et al. (1967)** — Eisenhower Expressway, Chicago (3-lane, 55 mph). The canonical benchmark dataset for comparing macroscopic models; 118 one-minute observations. Already in imperial units. Most widely cited parameter comparison in the literature.
- **Lu & Meng (2013)** — Two sites in China: Beijing Third Ring Road (6-lane urban, 80 km/h) and Jing Jin Tang Highway (4-lane intercity, 110 km/h). Higher free-flow speeds than SR-210 but valuable for cross-study spread.
- **Romanowska & Jamroz (2021)** — S6 Expressway, Poland (4-lane, 120 km/h). Largest dataset (37.5M vehicles, 36 months). Provides Greenshields and Underwood only (excluded Greenberg due to boundary condition failure).
- **Tiwari & Marsani (2014)** — Nepal hilly terrain (two-lane undivided, mixed traffic). **Closest road-type analog to SR-210** — two-lane, steep, curved. However, the Greenberg $k_j = 1452$ veh/km (→ ~2338 veh/mi after conversion) is a known outlier, almost certainly caused by mixed-traffic counting that includes non-motorized vehicles, inflating observed density far beyond what a US highway would produce.

All metric values are converted to mph and veh/mi at registry time.

In [ ]:
# --- raw literature registry (before unit conversion) ---
LITERATURE_RAW = {
    "greenshields": [
        {"source": "Drake_1967_Eisenhower",   "v_f": 58.6,  "k_j": 125.0, "units": "imperial", "n_obs": 118},
        {"source": "Lu_Meng_2013_Beijing",     "v_f": 80.3,  "k_j": 114.2, "units": "metric",   "n_obs": 88},
        {"source": "Lu_Meng_2013_JJT",         "v_f": 118.6, "k_j": 54.1,  "units": "metric",   "n_obs": 88},
        {"source": "Romanowska_2021_S6",        "v_f": 121.0, "k_j": 138.0, "units": "metric",   "n_obs": 500},
        {"source": "Tiwari_2014_Nepal",         "v_f": 62.9,  "k_j": 150.0, "units": "metric",   "n_obs": 48},
    ],
    "greenberg": [
        {"source": "Drake_1967_Eisenhower",   "v_0": 32.8,  "k_j": 146.0,  "units": "imperial", "n_obs": 118},
        {"source": "Lu_Meng_2013_Beijing",     "v_0": 30.8,  "k_j": 143.9,  "units": "metric",   "n_obs": 88},
        {"source": "Lu_Meng_2013_JJT",         "v_0": 47.0,  "k_j": 83.1,   "units": "metric",   "n_obs": 88},
        {"source": "Tiwari_2014_Nepal",         "v_0": 12.9,  "k_j": 1452.0, "units": "metric",   "n_obs": 48,
         "outlier": True,
         "note": "k_j extremely high (~2338 veh/mi after conversion); likely mixed-traffic artifact"},
    ],
    "underwood": [
        {"source": "Drake_1967_Eisenhower",   "v_f": 76.8,  "k_0": 56.9,  "units": "imperial", "n_obs": 118},
        {"source": "Lu_Meng_2013_Beijing",     "v_f": 85.6,  "k_0": 57.5,  "units": "metric",   "n_obs": 88},
        {"source": "Lu_Meng_2013_JJT",         "v_f": 134.3, "k_0": 29.0,  "units": "metric",   "n_obs": 88},
        {"source": "Romanowska_2021_S6",        "v_f": 141.0, "k_0": 79.0,  "units": "metric",   "n_obs": 500},
        {"source": "Tiwari_2014_Nepal",         "v_f": 64.9,  "k_0": 118.0, "units": "metric",   "n_obs": 48},
    ],
}




LIT = ebh.convert_literature_entries(LITERATURE_RAW)
ebh.display_entries(LIT)

---
## 4. Sensitivity Analysis

Before committing to the literature parameter ranges for benchmarking, it is important to understand how the shape of each model responds to parameter variation. Each figure below holds one parameter fixed at its mid-range value and sweeps the other parameter across a physically plausible range. **Color encodes the swept parameter magnitude** (viridis scale).

These plots answer questions like: *Does increasing $k_j$ in Greenshields primarily shift the jam density or also the speed-at-capacity? How does changing $k_0$ in Underwood alter the shape of speed decline at low vs. high densities?* Identifying the operating regime of SR-210 (estimated low-to-moderate density, $k < 30$ veh/mi for most hours) helps focus on the parameter region that matters most for model validation.

In [ ]:
# --- sensitivity parameter grids ---

# Greenshields Model Perams
GS_VF_GRID  = np.linspace(34, 37, 7)    # mph
GS_KJ_GRID  = np.linspace(80, 170, 7)   # veh/mi
GS_KJ_FIXED = 125.0
GS_VF_FIXED = 35.0

# Greenberg Model Perams
GR_V0_GRID  = np.linspace(15, 50, 7)    # mph
GR_KJ_GRID  = np.linspace(90, 200, 7)   # veh/mi
GR_KJ_FIXED = 130.0
GR_V0_FIXED = 28.0

# Underwood Model Perams
UW_VF_GRID  = np.linspace(34, 37, 7)    # mph
UW_K0_GRID  = np.linspace(10, 80, 7)    # veh/mi
UW_K0_FIXED = 25.0
UW_VF_FIXED = 35.0




In [ ]:
# --- Figure: Greenshields sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=FIG_W_WIDE)
ebh.sensitivity_plot_1param(
    axes[0], ebh.greenshields,
    GS_VF_GRID, {"k_j": GS_KJ_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="viridis", sr210_range=(34, 37),
    title=f"Greenshields: Free-Flow Speed sweep  (Jam Density = {GS_KJ_FIXED:.0f} veh/mi fixed)"
)
ebh.sensitivity_plot_1param(
    axes[1], ebh.greenshields,
    GS_KJ_GRID, {"v_f": GS_VF_FIXED},
    swept_param_name="k_j", param_units=" veh/mi",
    cmap_name="plasma", sr210_range=(34, 37),
    title=f"Greenshields: Jam Density sweep  (Free-Flow Speed = {GS_VF_FIXED:.0f} mph fixed)"
)
plt.suptitle("Greenshields (1935) — Parameter Sensitivity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_greenshields.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: Greenberg sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=FIG_W_WIDE)
ebh.sensitivity_plot_1param(
    axes[0], ebh.greenberg,
    GR_V0_GRID, {"k_j": GR_KJ_FIXED},
    swept_param_name="v_0", param_units=" mph",
    cmap_name="viridis",
    title=f"Greenberg: Speed Constant sweep  (Jam Density = {GR_KJ_FIXED:.0f} veh/mi fixed)"
)
axes[0].set_ylim(0, 110)   # Greenberg can be large at low k
ebh.sensitivity_plot_1param(
    axes[1], ebh.greenberg,
    GR_KJ_GRID, {"v_0": GR_V0_FIXED},
    swept_param_name="k_j", param_units=" veh/mi",
    cmap_name="plasma",
    title=f"Greenberg: Jam Density sweep  (Speed Constant = {GR_V0_FIXED:.0f} mph fixed)"
)
axes[1].set_ylim(0, 110)
plt.suptitle("Greenberg (1959) — Parameter Sensitivity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_greenberg.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: Underwood sensitivity ---
fig, axes = plt.subplots(1, 2, figsize=FIG_W_WIDE)
ebh.sensitivity_plot_1param(
    axes[0], ebh.underwood,
    UW_VF_GRID, {"k_0": UW_K0_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="viridis", sr210_range=(34, 37),
    title=f"Underwood: Free-Flow Speed sweep  (Optimal Density = {UW_K0_FIXED:.0f} veh/mi fixed)"
)
ebh.sensitivity_plot_1param(
    axes[1], ebh.underwood,
    UW_K0_GRID, {"v_f": UW_VF_FIXED},
    swept_param_name="k_0", param_units=" veh/mi",
    cmap_name="plasma", sr210_range=(34, 37),
    title=f"Underwood: Optimal Density sweep  (Free-Flow Speed = {UW_VF_FIXED:.0f} mph fixed)"
)
plt.suptitle("Underwood (1961) — Parameter Sensitivity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_underwood.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Figure: Combined sensitivity (v_f sweep for each model, side by side) ---
fig, axes = plt.subplots(1, 3, figsize=FIG_W_TRIPLE, sharey=True)

ebh.sensitivity_plot_1param(
    axes[0], ebh.greenshields,
    GS_VF_GRID, {"k_j": GS_KJ_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="Blues_r", sr210_range=(34, 37),
    title="Greenshields"
)
ebh.sensitivity_plot_1param(
    axes[1], ebh.greenberg,
    GR_V0_GRID, {"k_j": GR_KJ_FIXED},
    swept_param_name="v_0", param_units=" mph",
    cmap_name="Reds_r",
    ylabel="",
    title="Greenberg"
)
ebh.sensitivity_plot_1param(
    axes[2], ebh.underwood,
    UW_VF_GRID, {"k_0": UW_K0_FIXED},
    swept_param_name="v_f", param_units=" mph",
    cmap_name="Greens_r", sr210_range=(34, 37),
    ylabel="",
    title="Underwood"
)
axes[1].set_ylim(0, 95)

plt.suptitle("Model Shape Comparison — Free-Flow Speed Sweep (v_f / v_0)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("notebooks/figures/sensitivity_combined.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 5. Literature CI Bands (Monte Carlo)

These figures are the primary visualization output of the notebook. Rather than plotting a single "best estimate" curve from any one study, we treat the spread of published parameter values as **genuine uncertainty** about what parameters apply to a facility like SR-210. This is methodologically appropriate because none of the five studies was calibrated on a mountain two-lane highway — each represents an analogical reference, not a direct measurement.

**Methodology:** For each parameter, we fit a Gaussian distribution to the cross-study estimates (excluding the Tiwari Nepal Greenberg $k_j$ outlier). We then draw $N = 2000$ samples from the joint parameter distribution, compute a speed-density curve for each sample, and take the 2.5th and 97.5th percentiles of this family as the 95% CI band. The mean curve reflects the literature consensus. Individual study curves appear as thin dashed lines for transparency.

Three sets of fundamental diagrams are produced: **speed-density** ($k$–$v$), **flow-density** ($k$–$q$), and **speed-flow** ($v$–$q$). Each triple panel shows Greenshields, Greenberg, and Underwood side by side for direct comparison.

In [ ]:
dists = ebh.lit_param_dists(LIT)


fig, axes = ebh.plot_param_dists([
    ("v_f  (mph)",      dists["greenshields"]["v_f"], LIT["greenshields"]),
    ("k_j  (veh/mi)",   dists["greenshields"]["k_j"], [e for e in LIT["greenshields"] + LIT["greenberg"] if not e.get("outlier")]),
    ("v_0  (mph)",      dists["greenberg"]["v_0"], [e for e in LIT["greenberg"] if not e.get("outlier")]),
    ("k_0  (veh/mi)",   dists["underwood"]["k_0"], LIT["underwood"]),
])

fig, axes = ebh.plot_kv_ci_figure(dists, k_range=K_PLOT)


In [ ]:
# SR-210 engineering estimate ranges (HCM Chapter 15 + German mountain-road research)
SR210_ranges_engineering = {
    "v_f":       (34.0, 60.0),   # free-flow speed - i took these number from google maps late night traveltime 20-22 mins 
    "k_j":     (50.0, 250.0), # jam density
    "capacity_vehhr":(600.0, 2000.0),# one-directional capacity
    "k_0":     (15.0, 40.0),   # optimum density (Underwood)
    "v_0":       (15.0, 40.0),   # speed at max flow (Greenberg)
}

SR210_ranges_NCHRP = {
    "v_f":       (35.0, 60.0),   # free-flow speed - i took these number from google maps late night traveltime 20-22 mins 
    "k_j":     (40.0, 150.0), # jam density
    "capacity_vehhr":(700.0, 1400.0),# one-directional capacity
    "k_0":     (15.0, 50.0),   # optimum density (Underwood)
    "v_0":       (10.0, 35.0),   # speed at max flow (Greenberg)
}
fig, axes = ebh.plot_engineering_dists(SR210_ranges_NCHRP)
fig, axes = ebh.plot_kv_ci_from_engineering(SR210_ranges_NCHRP)


---
## 7. ABM Validation — Simulation vs Literature

This section loads Tier 2 spatial data from a validation sweep, computes instantaneous
snapshot-based density and speed per spatial bin, fits the three classical models via NLS,
and overlays the simulation-derived CI bands against the literature CI bands from Section 5.

**Snapshot density method:** At each sampled timestep, vehicles physically on the road
(0 ≤ distance_traveled ≤ road_length, status = driving/slowing) are binned into `n_spatial`
segments. Density = vehicles / segment_length (veh/mi), speed = arithmetic mean of vehicle
speeds in the bin. This avoids Edie's time-area denominator issues when most vehicles are
queued off-road (negative distance_traveled).

### === 7.1  Process validation sweep ===


In [ ]:

SWEEP_RESULTS = "data/outputs/sweeps/sweep_20260418_151828_results.parquet"
SEASONS_DIR   = "data/outputs/seasons"
ROAD_LENGTH_M = 19950.0
N_SPATIAL     = 20

kv_df = ebh.process_sweep_validation(
    SWEEP_RESULTS, 
    SEASONS_DIR,
    road_length_m=ROAD_LENGTH_M, 
    n_spatial=N_SPATIAL)

display(kv_df.head())
kv_df.speed_limit.value_counts()

#fig, axes = ebh.plot_kv_scatter_fit(kv_df)


In [ ]:
fig, axes = ebh.plot_kv_scatter_fit(kv_df, n_bins=50, n_mc=100)


### === 7.2  Aggregate fitted params into distributions ===


In [ ]:

sim_params, sim_curves = ebh.fit_kv_models(kv_df, k_range=K_PLOT)
sim_params

In [ ]:
lit_ci_engineering = ebh.ci_bands_from_ranges(SR210_ranges_engineering, k_range=K_PLOT)
lit_ci_NCHRP = ebh.ci_bands_from_ranges(SR210_ranges_NCHRP, k_range=K_PLOT)


### === 7.3  Build CI bands and comparison figure ===


In [ ]:
lit_ci_NCHRP = ebh.ci_bands_from_ranges(SR210_ranges_NCHRP, k_range=K_PLOT)
lit_ci = ebh.ci_bands_from_dists(dists, k_range=K_PLOT)

fig, axes = ebh.plot_kv_ci_comparison(K_PLOT, lit_ci_NCHRP, sim_curves)

In [ ]:
fig, axes = ebh.plot_kv_scatter_ci(lit_ci_NCHRP, kv_df, n_bins=10, k_range=K_PLOT)


---
### 7.4 Findings

**Fitted parameters (NLS on pooled sweep data):**

| Model | Parameter | Fitted Value | NCHRP Range | Within Range? | R² |
|---|---|---|---|---|---|
| Greenshields | v_f | 42.6 mph | 35–60 mph | Yes | 0.907 |
| Greenshields | k_j | 88.7 veh/mi | 40–150 veh/mi | Yes | |
| Greenberg | v_0 | 14.7 mph | 10–35 mph | Yes (low end) | 0.943 |
| Greenberg | k_j | 156.7 veh/mi | 40–150 veh/mi | Slightly above | |
| Underwood | v_f | 51.5 mph | 35–60 mph | Yes | 0.971 |
| Underwood | k_0 | 44.0 veh/mi | 15–50 veh/mi | Yes | |

**Key observations:**

1. **Underwood fits best (R² = 0.971).** The exponential decay captures SR-210's behavior across the full observed density range. The gentle asymptotic tail matches the simulation's gradual speed reduction under congestion — vehicles slow but rarely reach a full stop on this facility.

2. **Greenberg fits the congested regime well (R² = 0.943) but overpredicts speed at low density.** The scatter plot (cell 20) shows the Greenberg curve lifting above the data cloud at k < 10 veh/mi — the expected log-singularity artifact. The fitted v_0 = 14.7 mph sits at the low end of the NCHRP-informed range (10–35 mph), consistent with SR-210's steep grades forcing low speeds at capacity.

3. **Greenshields is the weakest fit (R² = 0.907).** The linear model cannot reproduce the concave speed-density relationship visible in the data — it overestimates speed at moderate densities (~20–50 veh/mi) and underestimates the rate of speed decline at high densities. This is consistent with NCHRP 17-65's finding that the two-lane highway speed-flow relationship is concave-up, not linear (Section 1.2.1, p.2; Section 4.1.2, p.91).

4. **The ABM curve falls within or near the NCHRP CI bands for all three models** (cell 25). For Greenshields and Underwood, the ABM fitted curve sits at the lower edge of the literature CI — expected because the literature studies are predominantly from flat, higher-speed facilities (freeways and expressways), while SR-210 has lower free-flow speeds and reaches capacity at lower densities.

5. **The Greenberg ABM curve falls below the literature CI band.** The literature mean is dominated by flat-facility studies (Drake: 32.8 mph, Lu & Meng JJT: 29.2 mph) where capacity speed is much higher. The ABM's v_0 = 14.7 mph reflects the grade-constrained reality of SR-210 where truck crawl speeds and no-passing zones compress the traffic stream.

6. **Speed limit stratification is visible in the scatter.** The `speed_limit` column shows four distinct populations (25, 30, 40, 50 mph). The single-model fit averages across these regimes. A segment-stratified fit would likely improve R² for all three models and is a natural next step.

7. **Greenberg k_j = 156.7 veh/mi slightly exceeds the NCHRP range ceiling of 150.** This is a known limitation of the Greenberg model on data that includes low-density observations — the optimizer pushes k_j higher to compensate for the log-singularity. The Greenshields k_j = 88.7 veh/mi is a more physically meaningful jam density estimate for SR-210.


---
## 8. Summary

Three classical speed-density models were fitted to SR-210 ABM simulation data from a validation sweep (891,444 spatial bin observations across multiple demand levels and random seeds).

**Model ranking by goodness of fit:** Underwood (R² = 0.971) > Greenberg (R² = 0.943) > Greenshields (R² = 0.907). The exponential Underwood model best captures SR-210's speed-density relationship, while the linear Greenshields model is too rigid to represent the concave shape observed in both the simulation and NCHRP empirical findings.

**All fitted parameters fall within or near the NCHRP-informed engineering ranges**, confirming that the ABM produces macroscopically plausible traffic dynamics. The simulation's lower free-flow speeds and earlier capacity onset (relative to literature means) are consistent with SR-210's extreme geometry — steep grades, tight curves, and no-passing constraints not represented in any of the five calibration studies.

**Limitations and next steps:**
- These are single-regime equilibrium models fitted to pooled data. SR-210 exhibits multi-regime behavior (speed-limit stratification, grade-induced platoons, hysteresis). Segment-stratified or direction-stratified fitting would improve accuracy.
- The Greenberg model is physically meaningful only in the congested regime (k > ~10 veh/mi). Its fitted v_0 = 14.7 mph should be interpreted as SR-210's speed at capacity, not as a free-flow parameter.
- No classical model can represent the capacity drop, queue spillback, or mode-choice feedback loops that the ABM is designed to study. The classical fits serve as a sanity check, not a replacement for the agent-based dynamics.
